<a href="https://colab.research.google.com/github/piaseckazaneta/Working_Dogs_Foundation/blob/main/Mapa_schronisk_dzialania_statystyki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# 1. INSTALACJA I IMPORTY
# ==========================================
# !pip install folium pandas openpyxl plotly branca

import pandas as pd
import folium
import plotly.express as px
import plotly.offline as po
from branca.element import Element
from google.colab import files

# ==========================================
# 2. WCZYTYWANIE I PRZYGOTOWANIE DANYCH
# ==========================================

# Interaktywne wgrywanie plików do Colab
print("Wybierz pliki: schroniska_statystyki_geo.xlsx oraz schroniska_polska_geo.xlsx")
uploaded = files.upload()

df_stats = pd.read_excel("schroniska_statystyki_geo.xlsx", sheet_name="statystyki")
df_adopsiaki = pd.read_excel("schroniska_statystyki_geo.xlsx", sheet_name="lokalizacje_adopsiaki")
df_firmy = pd.read_excel("schroniska_statystyki_geo.xlsx", sheet_name="lokalizacje_warsztatow_firmy")
df_schroniska = pd.read_excel("schroniska_polska_geo.xlsx", sheet_name="Schroniska w Polsce")

# Czyszczenie nazw kolumn
df_stats.columns = df_stats.columns.str.strip()
df_adopsiaki.columns = df_adopsiaki.columns.str.strip()
df_firmy.columns = df_firmy.columns.str.strip()
df_schroniska.columns = df_schroniska.columns.str.strip()

# Standaryzacja kolumny 'rok' (Twoja sprawdzona metoda)
for df in [df_stats, df_adopsiaki]:
    df.columns = [c.lower() if c.strip().lower() == 'rok' else c for c in df.columns]

df_stats["rok"] = df_stats["rok"].astype(str)

# Słownik mapujący nazwy kolumn na ładne etykiety osi Y
nazwy_osi_y = {
    "zgloszenia_adopsiaki": "Liczba zgłoszeń",
    "rodziny_adopsiaki": "Liczba rodzin",
    "dzieci_warsztaty": "Liczba osób",
    "placowki_warsztaty": "Liczba placówek",
    "firmy_dfc": "Liczba firm",
    "wolontariusze": "Liczba osób",
    "linkedin": "Liczba obserwujących",
    "facebook": "Liczba obserwujących",
    "instagram": "Liczba obserwujących",
    "liczba_darczyncow": "Liczba darczyńców"
}

# ==========================================
# 3. KONFIGURACJA MAPY (FOLIUM)
# ==========================================
mapa = folium.Map(location=[52.0, 19.5], zoom_start=6, tiles=None)
folium.TileLayer("CartoDB dark_matter", control=False).add_to(mapa)

def get_header(text):
    return f'<b class="custom-header" style="font-size: 15px; color: #00ffa2; text-transform: uppercase;">{text}</b>'

def get_symbol(color):
    return f'<i style="background-color:{color}; width: 14px; height: 14px; border-radius: 50%; display: inline-block; margin-right: 10px; border: 2px solid white; vertical-align: middle;"></i>'

# Warstwy mapy
folium.FeatureGroup(name=get_header("Legenda")).add_to(mapa)
warstwa_schroniska = folium.FeatureGroup(name=get_symbol("#517494") + "Schroniska w Polsce", show=False)
for _, row in df_schroniska.iterrows():
    if row["Lat (WGS84)"] == "BRAK": continue
    folium.CircleMarker(location=[row["Lat (WGS84)"], row["Lon (WGS84)"]], radius=5, color="white", weight=1, fill=True, fill_color="#517494", fill_opacity=0.5, popup=row["Nazwa schroniska"]).add_to(warstwa_schroniska)
warstwa_schroniska.add_to(mapa)

folium.FeatureGroup(name=get_header("Działalność Fundacji WDF")).add_to(mapa)
warstwa_firmy = folium.FeatureGroup(name=get_symbol("#ed4c3f") + "Warsztaty w firmach", show=True)
for _, row in df_firmy.iterrows():
    folium.CircleMarker(location=[row["Lat (WGS84)"], row["Lon (WGS84)"]], radius=8, color="white", weight=2, fill=True, fill_color="#ed4c3f", fill_opacity=0.9, popup=row["miasto"]).add_to(warstwa_firmy)
warstwa_firmy.add_to(mapa)

for r in [2023, 2024, 2025, 2026]:
    warstwa = folium.FeatureGroup(name=get_symbol("#0fa589") + f"Adopsiaki {r}", show=True)
    df_rok = df_adopsiaki[df_adopsiaki["rok_od"] <= r]
    for _, row in df_rok.iterrows():
        folium.CircleMarker(location=[row["Lat (WGS84)"], row["Lon (WGS84)"]], radius=6, color="white", weight=1, fill=True, fill_color="#0fa589", fill_opacity=0.7, popup=f"{row['miasto']} ({row['rok_od']})").add_to(warstwa)
    warstwa.add_to(mapa)

folium.LayerControl(collapsed=False).add_to(mapa)

css_dark = Element("<style>.leaflet-control-layers { width: 280px !important; background: rgba(30, 30, 30, 0.9) !important; color: #fff !important; border-radius: 12px !important; padding: 15px !important; } .leaflet-control-layers-overlays label:has(.custom-header) input { display: none !important; } .leaflet-control-layers-selector { accent-color: #0fa589; }</style>")
mapa.get_root().header.add_child(css_dark)

# ==========================================
# 4. GENEROWANIE WYKRESÓW (POPRAWKA UCINANIA)
# ==========================================
def zrob_wykres(df, kolumna, tytul):
    df_plot = df.copy()
    df_plot["etykieta"] = df_plot.apply(lambda row: str(row[kolumna]) + " *" if row["dane_czesciowe"] == 1 else str(row[kolumna]), axis=1)

    ladna_nazwa_y = nazwy_osi_y.get(kolumna, "Liczba")

    fig = px.bar(df_plot, x="rok", y=kolumna, text="etykieta", title=tytul,
                 labels={kolumna: ladna_nazwa_y, "rok": "Rok"})

    fig.update_traces(
        textposition="outside",
        marker_color="#16a085",
        cliponaxis=False  # Kluczowe: pozwala etykietom wystawać poza obszar kreślenia
    )

    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(showgrid=False, type='category'),
        # yaxis: rangemode="tozero" + extend pozwala na miejsce nad słupkami
        yaxis=dict(
            gridcolor="#f0f0f0",
            rangemode="tozero"
        ),
        # Dodajemy margines górny dla osi Y (np. 15% zapasu na etykiety)
        yaxis_range=[0, df_plot[kolumna].max() * 1.2],
        height=450, # Zwiększona wysokość, by zmieścić stopkę wykresu
        autosize=True,
        margin=dict(l=70, r=50, t=100, b=120), # Duże marginesy, by nic nie cięło
        title=dict(x=0.5, xanchor='center', y=0.95)
    )

    fig.add_annotation(
        text="* dane częściowe (I–II 2026)",
        xref="paper", yref="paper",
        x=1, y=-0.35, # Obniżona pozycja adnotacji
        showarrow=False,
        font=dict(size=10, color="gray"),
        align="right"
    )
    return po.plot(fig, include_plotlyjs=False, output_type='div')

# Ponowne wygenerowanie zmiennych (użyj tych samych nazw co wcześniej)
h_fig1 = zrob_wykres(df_stats, "zgloszenia_adopsiaki", "Liczba zgłoszeń #Adopsiaki")
h_fig2 = zrob_wykres(df_stats, "rodziny_adopsiaki", "Rodziny wsparte w #Adopsiaki")
h_fig3 = zrob_wykres(df_stats, "dzieci_warsztaty", "Przeszkolone dzieci")
h_fig4 = zrob_wykres(df_stats, "placowki_warsztaty", "Placówki warsztatowe")
h_fig5 = zrob_wykres(df_stats, "firmy_dfc", "Firmy w programie DFC")
h_fig6 = zrob_wykres(df_stats, "wolontariusze", "Wolontariusze")
h_fig7 = zrob_wykres(df_stats, "linkedin", "Obserwujący LinkedIn")
h_fig8 = zrob_wykres(df_stats, "facebook", "Obserwujący Facebook")
h_fig9 = zrob_wykres(df_stats, "instagram", "Obserwujący Instagram")
h_fig10 = zrob_wykres(df_stats, "liczba_darczyncow", "Darczyńcy cykliczni")

# ==========================================
# 5. BUDOWA DASHBOARDU HTML (Z POPRAWIONYM UKŁADEM I STOPKĄ)
# ==========================================
mapa_html = mapa._repr_html_()

html_dashboard = f"""
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="UTF-8">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Segoe UI', Tahoma, sans-serif; background: #eef2f5; color: #333; }}
        .header {{ background: #092e5a; color: white; padding: 20px 40px; display: flex; justify-content: space-between; align-items: center; }}
        .container {{ width: 1300px; max-width: 95%; margin: 0 auto; padding-bottom: 20px; }}
        .section-title {{ font-size: 18px; font-weight: bold; color: #092e5a; margin: 30px 0 15px; border-left: 5px solid #16a085; padding-left: 15px; }}
        .map-box {{ background: white; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 12px rgba(0,0,0,0.1); height: 500px; }}
        .tabs {{ display: flex; flex-wrap: wrap; gap: 10px; margin: 20px 0; }}
        .tab-btn {{ padding: 10px 20px; border: 2px solid #092e5a; border-radius: 20px; background: white; cursor: pointer; font-weight: bold; color: #092e5a; }}
        .tab-btn.active {{ background: #092e5a; color: white; }}

        .tab-content {{ display: none; grid-template-columns: repeat(2, 1fr); gap: 25px; width: 100%; }}
        .tab-content.active {{ display: grid; }}

        .chart-card {{ background: white; border-radius: 12px; padding: 15px; padding-bottom: 25px; box-shadow: 0 4px 10px rgba(0,0,0,0.05); overflow: hidden; }}

        .footer {{ background: #092e5a; color: white; text-align: center; padding: 40px 20px; margin-top: 50px; font-size: 14px; }}
        .footer a {{ color: #16a085; text-decoration: none; font-weight: bold; }}

        @media (max-width: 1000px) {{ .tab-content.active {{ grid-template-columns: 1fr; }} }}
    </style>
</head>
<body>
<div class="header">
    <div style="display: flex; align-items: center; gap: 20px;">
        <div style="background: white; padding: 5px 12px; border-radius: 4px; color: #092e5a; font-weight: bold;">WDF</div>
        <h1>Working Dogs Foundation</h1>
    </div>
    <div style="font-weight: bold;">2026</div>
</div>

<div class="container">
    <div class="section-title">Mapa Schronisk i Projektów</div>
    <div class="map-box">{mapa_html}</div>

    <div class="section-title">Statystyki Działań</div>
    <div class="tabs">
        <button class="tab-btn active" onclick="openTab('adopsiaki', this)">#Adopsiaki</button>
        <button class="tab-btn" onclick="openTab('warsztaty', this)">Warsztaty</button>
        <button class="tab-btn" onclick="openTab('dfc', this)">DFC</button>
        <button class="tab-btn" onclick="openTab('social', this)">Społeczność</button>
    </div>

    <div id="adopsiaki" class="tab-content active">
        <div class="chart-card">{h_fig1}</div><div class="chart-card">{h_fig2}</div>
    </div>
    <div id="warsztaty" class="tab-content">
        <div class="chart-card">{h_fig3}</div><div class="chart-card">{h_fig4}</div>
    </div>
    <div id="dfc" class="tab-content">
        <div class="chart-card">{h_fig5}</div><div class="chart-card">{h_fig10}</div>
    </div>
    <div id="social" class="tab-content">
        <div class="chart-card">{h_fig6}</div><div class="chart-card">{h_fig7}</div>
        <div class="chart-card">{h_fig8}</div><div class="chart-card">{h_fig9}</div>
    </div>
</div>

<div class="footer">
    <p>Dane: Working Dogs Foundation &copy; 2026 | Wspieramy schroniska i mądrą adopcję</p>
    <p style="margin-top: 10px;"><a href="https://workingdogs.pl/" target="_blank">Odwiedź naszą stronę: workingdogs.pl</a></p>
</div>

<script>
function openTab(id, btn) {{
    document.querySelectorAll('.tab-content').forEach(c => c.classList.remove('active'));
    document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
    document.getElementById(id).classList.add('active');
    btn.classList.add('active');
    // Wymuszenie przeliczenia szerokości wykresów po zmianie zakładki
    setTimeout(() => {{ window.dispatchEvent(new Event('resize')); }}, 50);
}}
</script>
</body>
</html>
"""

# Zapis i pobieranie
with open("dashboard_wdf_finalny.html", "w", encoding="utf-8") as f:
    f.write(html_dashboard)
files.download("dashboard_wdf_finalny.html")

FileNotFoundError: [Errno 2] No such file or directory: 'schroniska_statystyki_geo.xlsx'